# Lab 4 — Gọi Ollama local bằng OpenAI Python SDK

Notebook này chạy **100% local** qua `http://localhost:11434`. Không đăng nhập, không gọi cloud và không cần API key thật.

## Cách chạy

1. Chọn kernel Python của project: `.venv (Python 3.12.12)`.
2. Bấm **Run All**.
3. Lần chạy đầu, cell setup sẽ tự tải `llama3.2:3b` (khoảng 2 GB) nếu máy chưa có model.

## Mục tiêu

- Kiểm tra Ollama server và model local.
- Gọi endpoint tương thích OpenAI.
- Nhận response thường và streaming.
- Quan sát ảnh hưởng của `temperature`.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

MODEL = "llama3.2:3b"
OLLAMA_HOST = "http://localhost:11434"
OPENAI_BASE_URL = f"{OLLAMA_HOST}/v1"


def get_json(url: str) -> dict:
    with urlopen(url, timeout=10) as response:
        return json.load(response)


ollama_executable = shutil.which("ollama")
if ollama_executable is None:
    fallback = Path(os.environ.get("LOCALAPPDATA", "")) / "Programs" / "Ollama" / "ollama.exe"
    if fallback.is_file():
        ollama_executable = str(fallback)

if ollama_executable is None:
    raise RuntimeError("Không tìm thấy Ollama CLI. Hãy đóng/mở lại VS Code rồi chạy notebook lại.")

try:
    version = get_json(f"{OLLAMA_HOST}/api/version")["version"]
except (URLError, TimeoutError, KeyError) as exc:
    raise RuntimeError(
        "Ollama server chưa chạy. Hãy mở Ollama từ Start Menu rồi chạy lại cell này."
    ) from exc

installed_models = {item["name"] for item in get_json(f"{OLLAMA_HOST}/api/tags").get("models", [])}

if MODEL not in installed_models:
    print(f"Chưa có {MODEL}. Bắt đầu tải model — lần đầu có thể mất vài phút...")
    subprocess.run([ollama_executable, "pull", MODEL], check=True)
    installed_models = {item["name"] for item in get_json(f"{OLLAMA_HOST}/api/tags").get("models", [])}

if MODEL not in installed_models:
    raise RuntimeError(f"Đã chạy pull nhưng chưa tìm thấy {MODEL}. Kiểm tra output phía trên.")

print(f"✅ Ollama server: {version}")
print(f"✅ Model sẵn sàng: {MODEL}")
print(f"✅ Endpoint: {OPENAI_BASE_URL}")

## 4.1 — Response thông thường

Giá trị `api_key="ollama"` chỉ để OpenAI SDK chấp nhận cấu hình. Ollama local bỏ qua giá trị này.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=OPENAI_BASE_URL,
    api_key="ollama",
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "Bạn là gia sư AI. Trả lời rõ ràng, ngắn gọn bằng tiếng Việt.",
        },
        {
            "role": "user",
            "content": "Giải thích token, context window và quantization bằng ví dụ dễ hiểu.",
        },
    ],
    temperature=0.2,
)

answer = response.choices[0].message.content
print(answer)

## 4.2 — Streaming

Streaming in từng phần của câu trả lời ngay khi model sinh token, thay vì chờ toàn bộ response.

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Cho tôi 5 mẹo học Python hiệu quả, mỗi mẹo đúng một câu.",
        }
    ],
    temperature=0.2,
    stream=True,
)

for chunk in stream:
    text = chunk.choices[0].delta.content
    if text:
        print(text, end="", flush=True)

## 4.3 — Thí nghiệm temperature

Chạy cùng một prompt với `temperature=0.0` và `temperature=1.0`. Temperature cao làm output đa dạng/ngẫu nhiên hơn; nó không làm model thông minh hơn.

In [ ]:
def ask_with_temperature(temperature: float) -> str:
    result = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "Đặt 3 tên sáng tạo cho một ứng dụng học lập trình bằng AI.",
            }
        ],
        temperature=temperature,
    )
    return result.choices[0].message.content or ""


for value in (0.0, 1.0):
    print(f"\n{'=' * 20} temperature={value} {'=' * 20}")
    print(ask_with_temperature(value))

## Bài tập tự luyện

- [ ] Đổi system prompt để model đóng vai code reviewer.
- [ ] Đổi câu hỏi sang một chủ đề mày đang học.
- [ ] Chạy cell temperature ba lần và ghi lại điểm khác nhau.
- [ ] Chạy `ollama ps` và `nvidia-smi` trong terminal để xem model dùng VRAM.
- [ ] Sau đó thử model coding: `ollama pull qwen2.5-coder:3b`, rồi đổi biến `MODEL`.

### Hoàn thành lab khi

1. Preflight hiện ba dấu ✅.
2. Response thường in được câu trả lời.
3. Streaming hiển thị nội dung dần dần.
4. Mày giải thích được vai trò của model name, base URL, messages và temperature.